<a href="https://colab.research.google.com/github/Kelmoir/CAFA6_experimenting/blob/main/CAFA6_MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model building on the CAFA 6 datset, with the attempt on an MoE Architecture

This is just model building forbased on the CAFA 6 challenge from Kaggle: https://www.kaggle.com/competitions/cafa-6-protein-function-prediction/overview

In this notebook, I will use Huggingface transformers with the MOE approach

Basic exploration was done externally, thus won't be repeated in here.

The gist:

    GO factors are derived from proteins
    The list of them is usually not complete
    GO-terms are usually part of a graph like structure, a single one can't exist alone. See: https://geneontology.org/docs/ontology-documentation/
    82k proteis, 426k of possible GO-Terms
    This is a multi-label classification, thus The GO-terms will need to one-hot-encoded
    ProtBert model should provide base Protein understanding, more data augmeentation will need to happen on the GO-term side.

Provided data:
- train_sequences.fasta - amino acid sequences for proteins in the training set
- train_terms.tsv - the training set of proteins and corresponding annotated GO terms
- train_taxonomy.tsv - taxon IDs for proteins in the training set go-basic.obo - ontology graph structure
- testsuperset.fasta - amino acid sequences for proteins on which predictions should be made
- testsuperset-taxon-list.tsv - taxon IDs for proteins in the test
- superset IA.tsv - information accretion for each term (used to weight precision and recall)
- sample_submission.tsv - sample submission file in the correct format

Mixture of Experts article: https://medium.com/@rjnclarke/build-a-hard-mixture-of-experts-classifier-in-pytorch-b131cb0d2fa3
We are probably looking for a hard Setup here, with each expert supposedly only coveing a subtask

In Essency we build a Hirarcy of Models, and then split the decisions between them. The logical stuff will be handled in the model.forward Methods.

## 1. Setup work

### 1.1 Python dependencys

In [1]:
# Import the librays
try:
  import datasets
  #import gradio as gr
  import torchmetrics
  import pycocotools
  import evaluate
  import pandas as pd
  import obonet
  import numpy as np
  import torchinfo
  from Bio import SeqIO
except ModuleNotFoundError:
  !pip install datasets
  #!pip install gradio
  !pip install torchmetrics[detection]
  !pip install biopython
  !pip install obonet
  !pip install evaluate
  !pip install torchinfo

  import evaluate
  #import datasets
  #import gradio as gr

  # Required for evaluation
  import torchmetrics
  import pycocotools # make sure we got this for torchmetrics

  import pandas as pd
  import obonet
  import numpy as np
  from Bio import SeqIO

from torchinfo import summary
import random

import numpy as np

import torch
import transformers

from typing import List
from datasets import Value

MAX_LENGTH = 1024

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")
print(f"Using torchmetrics version: {torchmetrics.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
Using transformers version: 5.0.0
Using datasets version: 4.0.0
Using torch version: 2.10.0+cu128
Using torchmetrics version: 1.9.0


### 1.2 Data aquisition

temp model/etc go into runtime memory, long data is in github

Model upload to huggingface is just the most sensible solution for the good models

In [2]:
github_url = "https://github.com/Kelmoir/CAFA6_experimenting/raw/main/cafa-6-protein-function-prediction.zip"

import os
import requests
import zipfile
from pathlib import Path

data_path =Path("data/")
data_path.mkdir(exist_ok=True, parents=True)

if (data_path/"Train").is_dir():
  print("Data already there, skipping download")
else:
  print("Downlaoding data")
  with open(data_path/"cafa6.zip", "wb") as f:
    request = requests.get(github_url)
    print("Downloading data...")
    f.write(request.content)

  # unzip everything
  with zipfile.ZipFile(data_path/"cafa6.zip", "r") as zip_ref:
    print("unzipping files..")
    zip_ref.extractall(data_path)

Downlaoding data
unzipping files..


In [3]:
from pathlib import Path
demo_path = Path("../demos")
demo_path.mkdir(exist_ok=True)
model_data_path=demo_path / "model"
model_data_path.mkdir(exist_ok=True)
# We already got model_data_path
model_save_name = "CAFA6_classification__mixture_of_experts"
model_save_dir = Path(model_data_path, model_save_name)
model_save_dir

PosixPath('../demos/model/CAFA6_classification__mixture_of_experts')

### 1.3 Initial data aquisition

In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
prot_sequences ={}
unique_prot = []
for sequence in SeqIO.parse(data_path/"Train/train_sequences.fasta", "fasta"):
  unique_prot.append(sequence.id.split("|")[1])
  prot_sequences[sequence.id.split("|")[1]] = str(sequence.seq).upper().replace(r"[UZOB]", "X") # Reading the relevant data, and pre-formatting it for ProtBERT
train_df = pd.read_csv(data_path/"Train/train_terms.tsv", delimiter="\t")
obonet_graph = obonet.read_obo(data_path/"Train/go-basic.obo")
unique_go_terms = np.unique(train_df["term"])
unique_aspects = np.unique(train_df["aspect"])
# generate id2label and label2id datasets for the model to use (now for aspects)
id2label = {id: label for id, label in enumerate(unique_aspects)}
label2id = {label: id for id, label in enumerate(unique_aspects)}

In [ ]:
# Create a function to view a random sample
def print_random_item():
  rand_int= random.randint(0, len(unique_prot))
  key= unique_prot[rand_int]
  print(f"The Protein ID is: {key}")
  print(f" the sequence for it is: {prot_sequences[key]}")
  print(f" Available GO-terms:\n {train_df['term'][train_df['EntryID'] == key]}")

In [ ]:
print_random_item()

The Protein ID is: Q54I18
 the sequence for it is: MEPLRKRVKVYQLDNSGKWDDKGTGHVSCIYVDALCAMGLIVRSESDNSVILQTRLSAEDIYQKQQDSLIVWTEPDSQLDLALSFQDSLGCQDIWENILQYQNQRTGSCDSVDLDLPPVSINNLQTINELLEASLPMLDKDKIINSIFKEDLVRSLLDLFDEIEKSGEGGVHLFQIFNIFKNLILFNDTSILEVILSEDYLVRVMGALEYDPEISENNRIKHREFLNQQVVFKQVIKFPSKSLIGTIHQTFRIQYLKDVVLPRVLDDVTFSSLNSLIYFNNIDIVSQIQNDSDFLENLFSEIQKSEKNSEERKDLILFLQDLCNLAKGLQIQSKSTFFTVVVSLGLFKTLSAILDDENVQTRVSCTEIVLSTLLHDPEILRSYLCSPTSGNSKFLVQLINLFITDKDIGVKNQIVEIIKTLLEADSYDSSDFFRLFYDKGIDLLVSPLNEVYKGEPTIPGDPSSNLDSFVLYNIMELVIYCIKHHCYRIKHFIVEEGIAKKILRYTNPTGSGGGGGGGGNSERYLILGSIRFFRSMVNMKDDLYNQHIIQENLFEPIIEVFKSNISRYNLLNSAIIELFQYIYKENIRDLIVYLVERYRELFESVTYTDVLKQLILKYEQIKDSSFESPETSCNNNDSSSNDIDSKPIIGNNKINHNYQRTQREIDEEEEEAYFNRDDDSEDSDDEDELIPISINNNNNNNNNNKQICTNNENNMEKNDDNIEKDNENTNNGNGSSHIKIVDYEDEDDEDDEINKSVESDDIVEKHEIIDKNEKKDEIMKENNDSDNDDNDNNDNDNDNDNNSDIENKNHLNNNGNNENNENNDDVQDKSNNKNNSDKINEDEKIEKQDEMKENLEMEEIDEKVKEKQPKDIKKENQSQPDETVFNGKSNNSNNNNNNNNNNSNNQEIGDNRKTTPKRKLDYEKNESVVSKKIDKSNGPTSIDKDINGC

In [ ]:
train_df.head()

,EntryID,term,aspect
0,Q5W0B1,GO:0000785,C
1,Q5W0B1,GO:0004842,F
2,Q5W0B1,GO:0051865,P
3,Q5W0B1,GO:0006275,P
4,Q5W0B1,GO:0006513,P


### Exploration of aspects and the according GO-terms

In [ ]:
unique_aspects = np.unique(train_df["aspect"])
unique_aspect_specific_go_terms = {}
for item in unique_aspects:
  unique_aspect_specific_go_terms[item] =np.unique(train_df["term"][train_df["aspect"] == item])

In [ ]:
for key, value in unique_aspect_specific_go_terms.items():
  print(f"The experession aspect '{key}' has {len(value)} go terms associated to it.")

The experession aspect 'C' has 2651 go terms associated to it.
The experession aspect 'F' has 6616 go terms associated to it.
The experession aspect 'P' has 16858 go terms associated to it.


Analysis of the Approach, with regards to the aspects:

- The experession aspect 'C' has 2651 go terms associated to it.
- The experession aspect 'F' has 6616 go terms associated to it.
- The experession aspect 'P' has 16858 go terms associated to it.

This means, while the data is quite inbalanced, this approach should indeed work.

### Generating the first step Dataset

In [ ]:
train_df.columns.get_loc("EntryID")+1

1

In [11]:
from tqdm.auto import tqdm as progressbar

def generate_dataset(input_data: pd.DataFrame, label: str, constraint = lambda: True, prot_sequences = prot_sequences, max_length=1024)-> datasets.Dataset:
  """
  Generates a dataset from thr input data with the given constraints

  Args:
  input_data: Pandas Dataframe contraining the relevant data [ID, EntryID, labels_and_constraints]
  labels: str, marking the labels that you want to extract
  constraint: Any additional contraint that will be called for each sample, to see, if it should be included
  prot_sequences: The dictionary of proteins and their sequences {Protein: sequence}
  """
  label_idx = train_df.columns.get_loc(label)+1
  label_list = []
  sequences = []
  features = []
  last_feature = input_data["EntryID"][0]
  label_string = ""
  for item in progressbar(train_df.itertuples(), total=train_df.shape[0], desc="creating data as alligned lists"):
    if item[1] != last_feature:
      if len(prot_sequences[last_feature]) <= max_length:
        label_list.append(label_string)
        sequences.append(prot_sequences[last_feature])
        features.append(last_feature)
      last_feature = item[1]
      label_string = item[label_idx]
    elif item[label_idx] not in label_string:
      label_string += " " + item[label_idx]
  #handle last item
  if len(prot_sequences[last_feature]) <= max_length:
    label_list.append(label_string)
    sequences.append(prot_sequences[last_feature])
    features.append(last_feature)
  print(f"[INFO] Generating Dataset with {len(features)} items, with a label of {label}.")
  print(f"[INFO Number of proteins in the dataset: {len(features)}")
  print(f"[INFO] Number of proteins removed from data source: {len(prot_sequences) - len(features)}")
  dataset_dict = {"protein": features, "labels": label_list, "sequence": sequences}
  return datasets.Dataset.from_dict(dataset_dict)


In [41]:
from tqdm.auto import tqdm as progressbar

def generate_dataset(input_data: pd.DataFrame, label: str, entry_ids: list, prot_sequences = prot_sequences, max_length=1024)-> datasets.Dataset:
  """
  Generates a dataset for a specific subset of EntryIDs to prevent leakage.
  """
  # Filter dataframe to only include specific proteins
  subset_df = input_data[input_data['EntryID'].isin(entry_ids)].reset_index(drop=True)

  label_idx = subset_df.columns.get_loc(label)+1
  label_list = []
  sequences = []
  features = []

  # Group by EntryID to aggregate aspects/terms
  grouped = subset_df.groupby('EntryID')

  for entry_id, group in progressbar(grouped, desc=f"Processing {label}"):
    if entry_id in prot_sequences and len(prot_sequences[entry_id]) <= max_length:
      unique_labels = " ".join(group[label].unique())
      features.append(entry_id)
      sequences.append(prot_sequences[entry_id])
      label_list.append(unique_labels)

  dataset_dict = {"protein": features, "labels": label_list, "sequence": sequences}
  return datasets.Dataset.from_dict(dataset_dict)

In [ ]:
aspect_dataset = generate_dataset(train_df, label="aspect", max_length = MAX_LENGTH)
# Generate the actual dataset and save it to Google drive
dataset_save_path = model_data_path/f"aspect_max{MAX_LENGTH}"
aspect_dataset.save_to_disk(dataset_path=dataset_save_path)
print(f"[INFO] Stored the dataset in {dataset_save_path}, with a maximum protein_length of {MAX_LENGTH}")

TypeError: generate_dataset() missing 1 required positional argument: 'entry_ids'

In [9]:
aspect_dataset[8]

NameError: name 'aspect_dataset' is not defined

## 2. Helper functions

### 2.1 Tokenization

In [ ]:
from google.colab import userdata
from transformers import BertTokenizerFast
def create_tokenizer():
  return BertTokenizerFast.from_pretrained("Rostlab/prot_bert",
                                       do_lower_case=False,
                                       token=userdata.get("Huggingface"))
tokenizer = create_tokenizer()

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [42]:
from google.colab import userdata
from transformers import EsmTokenizer
def create_esm_tokenizer():
  # Instantiate EsmTokenizer using from_pretrained, similar to BertTokenizerFast
  # We'll use the same model as for the EsmForSequenceClassification later for consistency.
  return EsmTokenizer.from_pretrained("facebook/esm2_t30_150M_UR50D",
                                       token=userdata.get("Huggingface"))
tokenizer = create_esm_tokenizer()

In [57]:
#Tokenization test
rand_int= random.randint(0, len(unique_prot))
key= unique_prot[rand_int]
print(f"The Protein ID is: {key}")
print(f" the sequence for it is: {prot_sequences[key]}")
tokenized = tokenizer(prot_sequences[key])
print(f"The tokenized output: {tokenized}")
print(f"tokenized_keys: {[item for item in tokenized.keys()]}")

The Protein ID is: P09237
 the sequence for it is: MRLTVLCAVCLLPGSLALPLPQEAGGMSELQWEQAQDYLKRFYLYDSETKNANSLEAKLKEMQKFFGLPITGMLNSRVIEIMQKPRCGVPDVAEYSLFPNSPKWTSKVVTYRIVSYTRDLPHITVDRLVSKALNMWGKEIPLHFRKVVWGTADIMIGFARGAHGDSYPFDGPGNTLAHAFAPGTGLGGDAHFDEDERWTDGSSLGINFLYAATHELGHSLGMGHSSDPNAVMYPTYGNGDPQNFKLSQDDIKGIQKLYGKRSNSRKK
The tokenized output: {'input_ids': [0, 20, 10, 4, 11, 7, 4, 23, 5, 7, 23, 4, 4, 14, 6, 8, 4, 5, 4, 14, 4, 14, 16, 9, 5, 6, 6, 20, 8, 9, 4, 16, 22, 9, 16, 5, 16, 13, 19, 4, 15, 10, 18, 19, 4, 19, 13, 8, 9, 11, 15, 17, 5, 17, 8, 4, 9, 5, 15, 4, 15, 9, 20, 16, 15, 18, 18, 6, 4, 14, 12, 11, 6, 20, 4, 17, 8, 10, 7, 12, 9, 12, 20, 16, 15, 14, 10, 23, 6, 7, 14, 13, 7, 5, 9, 19, 8, 4, 18, 14, 17, 8, 14, 15, 22, 11, 8, 15, 7, 7, 11, 19, 10, 12, 7, 8, 19, 11, 10, 13, 4, 14, 21, 12, 11, 7, 13, 10, 4, 7, 8, 15, 5, 4, 17, 20, 22, 6, 15, 9, 12, 14, 4, 21, 18, 10, 15, 7, 7, 22, 6, 11, 5, 13, 12, 20, 12, 6, 18, 5, 10, 6, 5, 21, 6, 13, 8, 19, 14, 18, 13, 6, 14, 6, 17, 11, 4, 5, 21, 5, 18,

### 2.2 Preprocess batch

In [58]:
from functools import partial
def preprocess_batch(examples, tokenizer = tokenizer, transforms = None, max_length=1024, unique_items=unique_aspects):
  """
  Preprocesses a batch of items from the dataset, and gets them ready for model training.
  Removes padding from tokenizer calls, as padding will be handled by the data_collate_function.

  Parameters:
  **examples** a dataset items (dict of lists), which contain the preprocessed samples.
  **tokenizer** an instance of the ProtBERT tokenizer, in order to perform the tokenization.
  **transforms** Any data-augmeentation transform instructions. None available, atm.
  **max_length** The maximum sequence length for tokenization. Sequences longer than this will be truncated.

  Returns: preprocessed dataset items (dict of lists), ready for passing to the model
  """
  # Common part for both batched and single example processing
  sequences_processed = [item for item in examples["sequence"]] if isinstance(examples["sequence"], list) else [examples["sequence"]]

  # Tokenize with truncation, return as Python lists of integers
  tokenized_output = tokenizer(sequences_processed, max_length=max_length, truncation=True)

  # Process labels - if they are there.
  labels_list = []
  if "labels" in examples:
    go_term_lists_processed = examples["labels"] if isinstance(examples["labels"], list) else [examples["labels"]]

    for go_terms_str in go_term_lists_processed:
      labels = torch.full((len(unique_items),), 0, dtype=torch.float)
      for item in go_terms_str.split(" "):
        if item:
            labels[label2id[item]] = 1
      labels_list.append(labels)

  if isinstance(examples["protein"], list):
    output_dict = {
        "protein": examples["protein"],
        "input_ids": [torch.tensor(ids, dtype=torch.long) for ids in tokenized_output["input_ids"]],
        "attention_mask": [torch.tensor(ids, dtype=torch.long) for ids in tokenized_output["attention_mask"]]
    }
    # ESM models don't use token_type_ids, so we make it optional
    if "token_type_ids" in tokenized_output:
        output_dict["token_type_ids"] = [torch.tensor(ids, dtype=torch.long) for ids in tokenized_output["token_type_ids"]]

    if "labels" in examples:
        output_dict["labels"] = labels_list
    return output_dict
  else:
    output_dict = {
        "protein": examples["protein"],
        "input_ids": torch.tensor(tokenized_output["input_ids"][0], dtype=torch.long),
        "attention_mask": torch.tensor(tokenized_output["attention_mask"][0], dtype=torch.long)
    }
    if "token_type_ids" in tokenized_output:
        output_dict["token_type_ids"] = torch.tensor(tokenized_output["token_type_ids"][0], dtype=torch.long)

    if "labels" in examples:
        output_dict["labels"] = labels_list[0]
    return output_dict

preprocess_batch_partial = partial(preprocess_batch,
                                   tokenizer = tokenizer,
                                   transforms = None,
                                   max_length = MAX_LENGTH,
                                   unique_items=unique_aspects)

### 2.3 Data collation

In [66]:
from typing import List, Dict, Any
import torch

def data_collate_function(preprocessed_batch: List[Dict[str, Any]])-> Dict[str, Any]:
  """
  Stacks together groups of preprocessed samples into batches for our model,
  applying dynamic padding to the maximum sequence length within the batch.

  Params:
  **preprocessed_batch**: A list of Dicts, where each Dict represents a single processed sample
                          with unpadded 1D tensors for input_ids, token_type_ids, attention_mask, and labels.
  """
  collated_data = {}

  # Determine the maximum sequence length in the current batch
  max_seq_len = MAX_LENGTH

  # Manually pad each item in the batch
  padded_input_ids = []
  padded_token_type_ids = []
  padded_attention_mask = []
  labels = []

  for sample in preprocessed_batch:
    current_len = len(sample["input_ids"])
    padding_len = max_seq_len - current_len

    # Pad input_ids with the tokenizer's pad_token_id
    padded_input_ids.append(torch.cat([sample["input_ids"], torch.full((padding_len,), tokenizer.pad_token_id, dtype=torch.long)]))
    # Pad token_type_ids with 0
    if "token_tape_ids" in sample:
      padded_token_type_ids.append(torch.cat([sample["token_type_ids"], torch.full((padding_len,), 0, dtype=torch.long)]))
    # Pad attention_mask with 0
    padded_attention_mask.append(torch.cat([sample["attention_mask"], torch.full((padding_len,), 0, dtype=torch.long)]))
    if "labels" in sample:
      labels.append(sample["labels"])

  # Stack the padded tensors to form the batch
  collated_data["input_ids"] = torch.stack(padded_input_ids)
  if padded_token_type_ids != []:
    collated_data["token_type_ids"] = torch.stack(padded_token_type_ids)
  collated_data["attention_mask"] = torch.stack(padded_attention_mask)
  if labels != []:
    collated_data["labels"] = torch.stack(labels)

  return collated_data

### 2.4 Training arguments

Intially, I will fllow the general cooing book from mrDBurke: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial#setting-up-a-model-for-training

|Parameters | Value|
|-----------|------|
|`output_dir`| `model_save_dir`|
|`learning_rate`| 0.0001|
|`per_device_train_batch_size`| 32 |
|`per_device_eval_batch_size`| 32 |
|`num_train_epochs`| 10 |
|`eval_strategy` | "epoch" Evaluating per Epoch should prvide accurate results|
|`save_strategy` | "epoch" Dito|
|`save_total_limit` | 1 - The best model should suffice |
|`use_cpu`| "False" |
|`seed` | 42 |
|`load_best_model_at_end` | "True" Why bother about bad models?|
|`logging_strategy` | "epoch" No need for finer logging |
|`report_to` | "none" Local saving shal suffice, for now|
|`push_to_hub`| "False" pushing will be done manually, if there is anything of worth|

In [46]:
os.cpu_count()

2

In [47]:
# 3. Set up trainingArguments mit reduzierter Lernrate für Fine-Tuning
from transformers import TrainingArguments
import os
import datetime

BATCH_SIZE = 32
DATA_LOADER_NUM_WORKERS = 2
NUM_EPOCHS = 30

# Reduziert von 1e-3 auf 5e-5 für stabileres Fine-Tuning
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_STEPS = 50

DATE = datetime.date.today()
OUTPUT_DIR = Path(demo_path/f"CAFA6_classification_ESM_aspects")

training_args = TrainingArguments(
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    max_grad_norm = MAX_GRAD_NORM,
    warmup_steps = WARMUP_STEPS,
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs = NUM_EPOCHS,
    logging_strategy = "epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=torch.cuda.is_available(), # Nutze FP16 falls GPU vorhanden
    dataloader_num_workers=DATA_LOADER_NUM_WORKERS,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    push_to_hub=False,
    remove_unused_columns=False
)

### 2.5 Trainer

In [48]:
from transformers import Trainer
def create_trainer(model: torch.nn.Module,
                  dataset: datasets.Dataset,
                  data_factor:float=1) -> transformers.Trainer:
  """Sets up a Transformers Trainer for model training.

  Args:
    model (torch.nn.Module): The model to be trained.
    dataset (datasets.Dataset): The dataset containing 'train' and 'validation' splits.
    data_factor (float, optional): A factor to subsample the training and validation datasets. Defaults to 1.

  Returns:
    transformers.Trainer: An instance of the Hugging Face Trainer configured with the provided model, arguments, and datasets.
  """
  samples_train = int(len(dataset["train"])*data_factor)
  samples_validation = int(len(dataset["validation"])*data_factor)
  print(f"[INFO] Generating trainer with {samples_train} training items and {samples_validation} validation items.")
  trainer = Trainer(
      model = model,
      args = training_args,
      train_dataset = dataset["train"].select(range(samples_train)),
      eval_dataset = dataset["validation"].select(range(samples_validation)),
      data_collator = data_collate_function,
  )
  return trainer

### 2.6 Train_test_split, actual data generation...

In [67]:
# Create train, test, validation split
# Split the data into train and test splits
def create_train_val_split_from_dataset(dataset):
  # Shuffle first to ensure indices are randomized before splitting
  dataset = dataset.shuffle(seed=42)

  dataset_split = dataset.train_test_split(test_size=0.2, seed=42)

  # split the 80% train into train and validation (70/30 of the 80%)
  dataset_val_split = dataset_split["train"].train_test_split(test_size=0.3, seed=42)

  return datasets.DatasetDict({
      "train": dataset_val_split["train"],
      "validation": dataset_val_split["test"],
      "test": dataset_split["test"]
  })

In [68]:
# 1. Get unique protein IDs
unique_protein_ids = list(train_df['EntryID'].unique())
random.seed(42)
random.shuffle(unique_protein_ids)

# 2. Split IDs manually (80/20 split)
test_size = int(len(unique_protein_ids) * 0.2)
train_val_ids = unique_protein_ids[:-test_size]
test_ids = unique_protein_ids[-test_size:]

# 3. Split Train/Val IDs (70/30 of the 80%)
val_size = int(len(train_val_ids) * 0.3)
train_ids = train_val_ids[:-val_size]
val_ids = train_val_ids[-val_size:]

print(f"Splitting IDs: Train={len(train_ids)}, Val={len(val_ids)}, Test={len(test_ids)}")

# 4. Generate disjoint datasets
processed_dataset = datasets.DatasetDict({
    "train": generate_dataset(train_df, "aspect", train_ids),
    "validation": generate_dataset(train_df, "aspect", val_ids),
    "test": generate_dataset(train_df, "aspect", test_ids)
})

# 5. Apply transforms
processed_dataset["train"] = processed_dataset["train"].with_transform(transform=preprocess_batch_partial)
processed_dataset["validation"] = processed_dataset["validation"].with_transform(transform=preprocess_batch_partial)
processed_dataset["test"] = processed_dataset["test"].with_transform(transform=preprocess_batch_partial)

print("\nDisjoint datasets created successfully.")

Splitting IDs: Train=46147, Val=19777, Test=16480


Processing aspect:   0%|          | 0/46147 [00:00<?, ?it/s]

Processing aspect:   0%|          | 0/19777 [00:00<?, ?it/s]

Processing aspect:   0%|          | 0/16480 [00:00<?, ?it/s]


Disjoint datasets created successfully.


### 2.7 Output& Predictions

In [62]:
import torch
from torch import nn

def process_result(logits, threshold=0.6, top_n=10):
  """
  This function will take in a single protein logit prediction and turns it into alist of predicted GO-terms
  Args:
  **logits** a single Logit array
  **threshold** A float value between 0 and 1. Only GO-terms with a score higher than this value will be considered.
  **top_n** An integer value. If no GO-terms are above the threshold, the top_n GO-terms will be returned.
  Ensure, that the length is equal
  """
  all_results = []
  # Convert logits to probabilities using sigmoid
  probabilities = torch.sigmoid(logits).squeeze()

  # Get indices of GO terms above threshold
  above_threshold_indices = torch.where(probabilities > threshold)[0]

  # If no terms above threshold, take the top_n terms
  if len(above_threshold_indices) == 0:
    top_n_values, top_n_indices = torch.topk(probabilities, k=min(top_n, len(probabilities)))
    selected_indices = top_n_indices
    selected_probabilities = top_n_values
  else:
    selected_indices = above_threshold_indices
    selected_probabilities = probabilities[above_threshold_indices]

  # Sort selected terms by probability in descending order
  sorted_probabilities, sort_indices = torch.sort(selected_probabilities, descending=True)
  sorted_indices = selected_indices[sort_indices]

  for idx, prob in zip(sorted_indices, sorted_probabilities):
    go_term = id2label[idx.item()]
    score = prob.item()
    all_results.append([go_term, f"{score:.3f}"]) # Format score to 3 decimal places

  results_df = pd.DataFrame(all_results)
  # The request specifies 3 columns, all string, no headers, no index
  return results_df

def predict_on_input(input: str,
                     threshold:float,
                     model: nn.Module,
                     device:str = device):
  model.eval()
  with torch.inference_mode():
    tokenized = tokenizer(prepare_for_tokenization(input),
                          return_tensors="pt").to(device)
    output_logits = model(**tokenized)

  output_table = process_result(output_logits.logits,
                                threshold=threshold)

  return output_table

## 3. Model creation - Iterations

- We will try regault `nn.Module`-based solutions first.
- Max Input length will be accepted.
- We start with 5 hidden layers, and the work from there.

In [ ]:
from torch import nn
import torch.nn.functional as F
# First iteration, linear baseline
class ProtInteractionClassifierV0(nn.Module):
  def __init__(self, in_features: int, hidden_units: int, out_labels:int):
    super().__init__()
    self.linear_stack = nn.Sequential(
        nn.Linear(in_features=in_features,
                  out_features=hidden_units),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=hidden_units),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=hidden_units),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=hidden_units),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=out_labels),
    )
  def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
    # Cast input_ids to float before passing to nn.Linear
    logits = self.linear_stack(input_ids.float())

    if labels is not None:
      # Assuming multi-label classification, use BCEWithLogitsLoss
      loss = F.binary_cross_entropy_with_logits(logits, labels, reduction='mean') # Explicitly set reduction to 'mean'
      return (loss, logits) # Return loss and logits
    return logits # Otherwise just return logits

In [ ]:
aspect_dataset[0]

{'protein': 'Q5W0B1',
 'labels': ' C F P',
 'sequence': 'MAQTVQNVTLSLTLPITCHICLGKVRQPVICINNHVFCSICIDLWLKNNSQCPACRVPITPENPCKEIIGGTSESEPMLSHTVRKHLRKTRLELLHKEYEDEIDCLQKEVEELKSKNLSLESQIKTILDPLTLVQGNQNEDKHLVTDNPSKINPETVAEWKKKLRTANEIYEKVKDDVDKLKEANKKLKLENGGLVRENLRLKAEVDNRSPQKFGRFAVAALQSKVEQYERETNRLKKALERSDKYIEELESQVAQLKNSSEEKEAMNSICQTALSADGKGSKGSEEDVVSKNQGDSARKQPGSSTSSSSHLAKPSSSRLCDTSSARQESTSKADLNCSKNKDLYQEQVEVMLDVTDTSMDTYLEREWGNKPSDCVPYKDEELYDLPAPCTPLSLSCLQLSTPENRESSVVQAGGSKKHSNHLRKLVFDDFCDSSNVSNKDSSEDDISRSENEKKSECFSSPKTGFWDCCSTSYAQNLDFESSEGNTIANSVGEISSKLSEKSGLCLSKRLNSIRSFEMNRTRTSSEASMDAAYLDKISELDSMMSESDNSKSPCNNGFKSLDLDGLSKSSQGSEFLEEPDKLEEKTELNLSKGSLTNDQLENGSEWKPTSFFLLSPSDQEMNEDFSLHSSSCPVTNEIKPPSCLFQTEFSQGILLSSSHRLFEDQRFGSSLFKMSSEMHSLHNHLQSPWSTSFVPEKRNKNVNQSTKRKIQSSLSSASPSKATKS'}

In [ ]:
for key, value in processed_dataset["train"][0].items():
  if isinstance(value, torch.Tensor):
    print(f"{key}: {value.dtype}")
  else:
    print(f"{key}: {value}")

protein: O70405
input_ids: torch.int64
token_type_ids: torch.int64
attention_mask: torch.int64
labels: torch.float32


In [ ]:
model_0 = ProtInteractionClassifierV0(in_features=MAX_LENGTH,
                                      hidden_units=1024,
                                      out_labels=len(id2label))
trainer_0 = create_trainer(model=model_0, dataset=processed_dataset, data_factor=0.1)

[INFO] Generating trainer with 5681 training items and 827 validation items.


In [ ]:
trainer_0.train()

Epoch,Training Loss,Validation Loss
1,0.647473,0.614146
2,0.626967,0.617730
3,0.623521,0.603123
4,0.621544,0.600973
5,0.622745,0.603513
6,0.626516,0.615234
7,0.622112,0.625069
8,0.621341,0.599295
9,0.621064,0.605504
10,0.620110,0.602445


TrainOutput(global_step=3560, training_loss=0.6106898854287822, metrics={'train_runtime': 82.5048, 'train_samples_per_second': 1377.132, 'train_steps_per_second': 43.149, 'total_flos': 0.0, 'train_loss': 0.6106898854287822, 'epoch': 20.0})

In [ ]:
from torch import nn
import torch.nn.functional as F
# First iteration, linear baseline
class ProtInteractionClassifierV1(nn.Module):
  def __init__(self, in_features: int, hidden_units: int, out_labels:int):
    super().__init__()
    self.linear_stack = nn.Sequential(
        nn.Conv1d(in_features=in_features,
                  out_features=hidden_units,
                  kernel_size=10,
                  stride=1,
                  padding=0),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=hidden_units),
        nn.ELU(),
        nn.Conv1d(in_features=hidden_units,
                  out_features=hidden_units,
                  kernel_size=10,
                  stride=1,
                  padding=0),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=hidden_units),
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=out_labels),
        nn.Dropout1d(p=0.1)
    )
  def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
    # Cast input_ids to float before passing to nn.Linear
    logits = self.linear_stack(input_ids.float())

    if labels is not None:
      # Assuming multi-label classification, use BCEWithLogitsLoss
      loss = F.binary_cross_entropy_with_logits(logits, labels, reduction='mean') # Explicitly set reduction to 'mean'
      return (loss, logits) # Return loss and logits
    return logits # Otherwise just return logits

In [ ]:
model_1 = ProtInteractionClassifierV0(in_features=MAX_LENGTH,
                                      hidden_units=1024,
                                      out_labels=len(id2label))
trainer_1 = create_trainer(model=model_1, dataset=processed_dataset, data_factor=0.1)

[INFO] Generating trainer with 5681 training items and 827 validation items.


In [ ]:
trainer_1.train()

Epoch,Training Loss,Validation Loss
1,0.649277,0.632950
2,0.625408,0.616015
3,0.623418,0.603579
4,0.621248,0.601250
5,0.622667,0.602127
6,0.623724,0.620669
7,0.619089,0.642022
8,0.622197,0.598618
9,0.622673,0.603433
10,0.618574,0.604568


TrainOutput(global_step=3560, training_loss=0.6090884605150544, metrics={'train_runtime': 77.4741, 'train_samples_per_second': 1466.554, 'train_steps_per_second': 45.951, 'total_flos': 0.0, 'train_loss': 0.6090884605150544, 'epoch': 20.0})

### 2.3 Basing off the VDCNN

In [ ]:
from torch import nn
import torch.nn.functional as F
# First iteration, linear baseline
class ProtInteractionClassifierV2(nn.Module):
  def __init__(self, in_features: int, hidden_units: int, out_labels:int):
    super().__init__()
    def create_conv_block(in_features:int,
                          out_features:int,
                          kernel_size:int):
      return nn.Sequential(
        nn.Conv1d(in_features=in_features,
                  out_features=out_features,
                  kernel_size=kernel_size,
                  stride=1,
                  padding=kernel_size // 2),
        nn.Conv1d(in_features=out_features,
                  out_features=out_features,
                  kernel_size=kernel_size,
                  stride=1,
                  padding=kernel_size // 2),
        nn.Conv1d(in_features=out_features,
                  out_features=out_features,
                  kernel_size=kernel_size,
                  stride=1,
                  padding=kernel_size // 2),
        nn.MaxPooling(kernel_size=4,
                      stride=2)
        )
    self.conv_block_1 = create_conv_block(in_features=in_features,
                                          out_features=64,
                                          kernel_size=3)
    self.conv_block_2 = create_conv_block(in_features=in_features,
                                          out_features=128,
                                          kernel_size=3)
    self.conv_block_3 = create_conv_block(in_features=in_features,
                                          out_features=256,
                                          kernel_size=3)
    self.linear_stack = nn.Sequential(
        nn.ELU(),
        nn.Linear(in_features=hidden_units,
                  out_features=out_labels),
        nn.Dropout1d(p=0.1)
    )
  def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
    # Cast input_ids to float before passing to nn.Linear
    logits = self.linear_stack(self.conv_block_3(self.conv_block_2(self.conv_block_1(input_ids.float()))))

    if labels is not None:
      # Assuming multi-label classification, use BCEWithLogitsLoss
      loss = F.binary_cross_entropy_with_logits(logits, labels, reduction='mean') # Explicitly set reduction to 'mean'
      return (loss, logits) # Return loss and logits
    return logits # Otherwise just return logits

In [ ]:
model_2 = ProtInteractionClassifierV0(in_features=MAX_LENGTH,
                                      hidden_units=1024,
                                      out_labels=len(id2label))
trainer_2 = create_trainer(model=model_2, dataset=processed_dataset, data_factor=0.1)

[INFO] Generating trainer with 5681 training items and 827 validation items.


In [ ]:
trainer_2.train()

Epoch,Training Loss,Validation Loss
1,0.649277,0.632950
2,0.625408,0.616015
3,0.623418,0.603579
4,0.621248,0.601250
5,0.622667,0.602127
6,0.623724,0.620669
7,0.619089,0.642022
8,0.622197,0.598618
9,0.622673,0.603433
10,0.618574,0.604568


TrainOutput(global_step=3560, training_loss=0.6090884605150544, metrics={'train_runtime': 77.8182, 'train_samples_per_second': 1460.07, 'train_steps_per_second': 45.748, 'total_flos': 0.0, 'train_loss': 0.6090884605150544, 'epoch': 20.0})

#### 3.3.1

In [ ]:
from torch import nn
import torch.nn.functional as F
# First iteration, linear baseline
class ProtInteractionClassifierV2(nn.Module):
  def __init__(self,
               vocab_size: int,
               embedding_dim: int,
               in_features: int,
               hidden_units: int,
               out_labels:int):
    super().__init__()
    def create_conv_block(
                          in_channels:int,
                          out_channels:int,
                          kernel_size:int):
      return nn.Sequential(
        nn.Conv1d(in_channels=in_channels,
                  out_channels=out_channels,
                  kernel_size=kernel_size,
                  stride=1),
        nn.Conv1d(in_channels=out_channels,
                  out_channels=out_channels,
                  kernel_size=kernel_size,
                  stride=1),
        nn.ELU(),
        nn.Conv1d(in_channels=out_channels,
                  out_channels=out_channels,
                  kernel_size=kernel_size,
                  stride=1),
        nn.MaxPool1d(kernel_size=4,
                      stride=2),
        )
    self.input = nn.Sequential(
      nn.Embedding(vocab_size, embedding_dim),
      nn.LayerNorm(embedding_dim),
    )
    self.conv_block_1 = create_conv_block(in_channels=embedding_dim,
                                          out_channels=hidden_units//32,
                                          kernel_size=3)
    self.conv_block_2 = create_conv_block(in_channels=hidden_units//32,
                                          out_channels=hidden_units//16,
                                          kernel_size=3)
    self.conv_block_3 = create_conv_block(in_channels=hidden_units//16,
                                          out_channels=hidden_units//8,
                                          kernel_size=3)
    self.conv_block_4 = create_conv_block(in_channels=hidden_units//8,
                                          out_channels=hidden_units//4,
                                          kernel_size=3)
    self.conv_block_5 = create_conv_block(in_channels=hidden_units//4,
                                          out_channels=hidden_units//2,
                                          kernel_size=3)
    self.conv_block_6 = create_conv_block(in_channels=hidden_units//2,
                                          out_channels=hidden_units,
                                          kernel_size=3)
    # In_features = Batch_size * conv_block_3[out_channels]
    self.linear_stack = nn.Sequential(
        nn.Flatten(),
        nn.ELU(),
        nn.Dropout1d(p=0.2),
        nn.Linear(in_features=embedding_dim*8,  # Output format of the 3rd Conv Block
                  out_features=out_labels),
    )
  def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
    #Debug
    #print(f"Input shape: {input_ids.shape}")
    # input_ids shape: (batch_size, MAX_LENGTH) e.g., (32, 1024)
    embedded = self.input(input_ids) # (batch_size, MAX_LENGTH, embedding_dim) e.g., (32, 1024, 1024)

    # Permute for Conv1d: (batch_size, embedding_dim, MAX_LENGTH)
    embedded_permuted = embedded.permute(0, 2, 1) # e.g., (32, 1024, 1024)
    # print(f"Shape after input: {embedded_permuted.shape}")
    # temp = self.conv_block_1(embedded_permuted)
    # print(f"Shape after conv block 1: {temp.shape}")
    # temp = self.conv_block_2(temp)
    # print(f"Shape after conv block 2: {temp.shape}")
    # temp = self.conv_block_3(temp)
    # print(f"Shape after conv block: {temp.shape}")
    logits = self.linear_stack(self.conv_block_6(self.conv_block_5(self.conv_block_4(self.conv_block_3(self.conv_block_2(self.conv_block_1(embedded_permuted)))))))


    if labels is not None:
      # Assuming multi-label classification, use BCEWithLogitsLoss
      loss = F.binary_cross_entropy_with_logits(logits, labels, reduction='mean') # Explicitly set reduction to 'mean'
      return (loss, logits) # Return loss and logits
    return logits # Otherwise just return logits

In [ ]:
model = ProtInteractionClassifierV2(vocab_size=tokenizer.vocab_size,
                                      embedding_dim=MAX_LENGTH,
                                      in_features=MAX_LENGTH,
                                      hidden_units=1024,
                                      out_labels=len(id2label))
trainer = create_trainer(model=model, dataset=processed_dataset, data_factor=0.3)

[INFO] Generating trainer with 12561 training items and 5379 validation items.


In [ ]:
from torchinfo import summary
summary(model=model,
        input_size=(1, MAX_LENGTH), #example of (batch size, characters
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"],
        dtypes=[torch.long])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
ProtInteractionClassifierV2 (ProtInteractionClassifierV2)    [1, 1024]            [1, 3]               --                   True
├─Sequential (input)                                         [1, 1024]            [1, 1024, 1024]      --                   True
│    └─Embedding (0)                                         [1, 1024]            [1, 1024, 1024]      30,720               True
│    └─LayerNorm (1)                                         [1, 1024, 1024]      [1, 1024, 1024]      2,048                True
├─Sequential (conv_block_1)                                  [1, 1024, 1024]      [1, 32, 508]         --                   True
│    └─Conv1d (0)                                            [1, 1024, 1024]      [1, 32, 1022]        98,336               True
│    └─Conv1d (1)                                            [1, 32, 1022]        [1, 32, 10

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.610215,0.585107
2,0.604162,0.599811
3,0.599078,0.592601
4,0.598314,0.585095
5,0.596890,0.584855
6,0.593923,0.580986
7,0.593116,0.582846
8,0.592639,0.588240
9,0.590443,0.582620
10,0.590265,0.584508


TrainOutput(global_step=11790, training_loss=0.5775702038895184, metrics={'train_runtime': 753.1897, 'train_samples_per_second': 500.312, 'train_steps_per_second': 15.653, 'total_flos': 0.0, 'train_loss': 0.5775702038895184, 'epoch': 30.0})

AttributeError: 'ProtInteractionClassifierV2' object has no attribute 'destroy'

### 2.4 Embeddings and LTSM

the CNN are not well suited, so let's try embeddings ans LTSM

See https://damien0x0023.github.io/rnnExplainer/ for a basic layout, which we will use as a baseline in the rnn setting...

In [ ]:
from torch.nn import Module

from torch import nn
import torch.nn.functional as F
# Embeddings
class ProtInteractionClassifierV3(nn.Module):
  def __init__(self, vocab_size: int,
               embedding_dim: int,
               hidden_units: int,
               out_labels:int,
               num_lstm_layers: int = 1, # Added number of LSTM layers
               dropout_rate: float = 0.1):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.ayer_norm=nn.LayerNorm()
    self.lstm = nn.LSTM(input_size=embedding_dim, # input_size should be embedding_dim
                        hidden_size=hidden_units,
                        num_layers=num_lstm_layers,
                        batch_first=True) # Set batch_first to True for easier handling with transformers batching

    self.linear_stack = nn.Sequential(
        nn.ELU(),
        nn.Linear(in_features=hidden_units, # Input to Linear layer is the hidden_size of LSTM
                  out_features=out_labels),
        nn.Dropout(p=dropout_rate) # Changed from Dropout1d to Dropout for 2D input
    )

  def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
    # input_ids shape: (batch_size, sequence_length)
    embedded = self.embedding(input_ids) # Output: (batch_size, sequence_length, embedding_dim)

    # lstm_output: (batch_size, sequence_length, hidden_units * num_directions) if batch_first=True
    # hidden_state: (num_lstm_layers * num_directions, batch_size, hidden_units)
    # cell_state: (num_lstm_layers * num_directions, batch_size, hidden_units)
    lstm_output, (hidden_state, cell_state) = self.lstm(embedded)

    # Take the hidden state of the last LSTM layer
    # hidden_state[-1, :, :] has shape (batch_size, hidden_units)
    final_features = hidden_state[-1, :, :]

    logits = self.linear_stack(final_features)

    if labels is not None:
      # Assuming multi-label classification, use BCEWithLogitsLoss
      loss = F.binary_cross_entropy_with_logits(logits, labels, reduction='mean') # Explicitly set reduction to 'mean'
      return (loss, logits) # Return loss and logits
    return logits # Otherwise just return logits

In [ ]:
model_3 = ProtInteractionClassifierV3(vocab_size=tokenizer.vocab_size,
                                      embedding_dim=MAX_LENGTH,
                                      hidden_units=400,
                                      num_lstm_layers=2,
                                      out_labels=len(id2label))
trainer_3 = create_trainer(model=model_3, dataset=processed_dataset, data_factor=0.1)

[INFO] Generating trainer with 5493 training items and 800 validation items.


In [ ]:
model_3

ProtInteractionClassifierV3(
  (embedding): Embedding(30, 1024)
  (lstm): LSTM(1024, 400, num_layers=2, batch_first=True)
  (linear_stack): Sequential(
    (0): ELU(alpha=1.0)
    (1): Linear(in_features=400, out_features=3, bias=True)
    (2): Dropout(p=0.1, inplace=False)
  )
)

In [ ]:
trainer_3.train()

Epoch,Training Loss,Validation Loss
1,0.628541,0.608284
2,0.622324,0.602696
3,0.618500,0.601698
4,0.621456,0.605019
5,0.620556,0.604432
6,0.621423,0.601876
7,0.622812,0.605329
8,0.619806,0.605038
9,0.621502,0.603358
10,0.619385,0.604321


TrainOutput(global_step=3560, training_loss=0.6209866234425748, metrics={'train_runtime': 1480.3451, 'train_samples_per_second': 76.752, 'train_steps_per_second': 2.405, 'total_flos': 0.0, 'train_loss': 0.6209866234425748, 'epoch': 20.0})

### 3.5 esm to see, what is possible

In [53]:
from transformers import AutoModelForSequenceClassification

def create_esm(unfreeze_last_n=2):
  model = AutoModelForSequenceClassification.from_pretrained("facebook/esm2_t30_150M_UR50D",
                                    num_labels= len(id2label),
                                    id2label = id2label,
                                    label2id=label2id,
                                    token=userdata.get("Huggingface"))

  # Erst alles einfrieren
  for param in model.esm.parameters():
    param.requires_grad = False

  # Dann die letzten n Layer des Encoders wieder auftauen
  # ESM2-t30 hat 30 Layer (0 bis 29)
  if unfreeze_last_n > 0:
    for i in range(30 - unfreeze_last_n, 30):
      for param in model.esm.encoder.layer[i].parameters():
        param.requires_grad = True

  return model

model_esm = create_esm(unfreeze_last_n=2)
trainer_esm = create_trainer(model=model_esm, dataset=processed_dataset, data_factor=0.1)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[INFO] Generating trainer with 4187 training items and 1793 validation items.


In [ ]:
# print a summaray with torchinfo
from torchinfo import summary
summary(model=model_esm,
        input_size=(1, MAX_LENGTH), #example of (batch size, characters
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"],
        dtypes=[torch.long])

Layer (type (var_name))                                                     Input Shape          Output Shape         Param #              Trainable
EsmForSequenceClassification (EsmForSequenceClassification)                 [1, 1024]            [1, 3]               --                   Partial
├─EsmModel (esm)                                                            [1, 1024]            [1, 1024, 640]       601                  Partial
│    └─EsmEmbeddings (embeddings)                                           --                   [1, 1024, 640]       --                   False
│    │    └─Embedding (word_embeddings)                                     [1, 1024]            [1, 1024, 640]       (21,120)             False
│    └─EsmEncoder (encoder)                                                 [1, 1024, 640]       [1, 1024, 640]       --                   Partial
│    │    └─ModuleList (layer)                                              --                   --                   14

In [ ]:
num_layers = model_esm.esm.config.num_hidden_layers
embedding_size = model_esm.esm.config.hidden_size

print(f"Das Modell hat {num_layers} Layer.")
print(f"Die Embedding-Dimension beträgt {embedding_size}.")

# Beispiel: Dynamisches Auslesen der Layer-Struktur
print(f"Anzahl der Encoder-Layer in der ModuleList: {len(model_esm.esm.encoder.layer)}")

In [ ]:
trainer_esm.train()

Epoch,Training Loss,Validation Loss
1,0.621279,0.620012
2,0.606944,0.619533
3,0.601549,0.617567
4,0.595868,0.619520
5,0.589365,0.620438
6,0.581024,0.625452
7,0.570076,0.633279
8,0.559020,0.637288
9,0.548787,0.641917
10,0.542262,0.643261


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.

TrainOutput(global_step=1720, training_loss=0.5816174529319585, metrics={'train_runtime': 3812.0468, 'train_samples_per_second': 14.41, 'train_steps_per_second': 0.451, 'total_flos': 4.998888680398848e+16, 'train_loss': 0.5816174529319585, 'epoch': 10.0})

In [54]:
import numpy as np
from evaluate import load

clf_metrics = load("f1", config_name="multilabel")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()

    # Berechne F1 für jede Klasse und den Durchschnitt
    f1_micro = clf_metrics.compute(predictions=predictions, references=labels, average="micro")["f1"]
    f1_macro = clf_metrics.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

In [ ]:
# Frisches Modell und Trainer mit Fokus auf Generalisierung
model_esm = create_esm(unfreeze_last_n=2)

trainer_esm_final = Trainer(
    model=model_esm,
    args=training_args,
    train_dataset=processed_dataset["train"].select(range(int(len(processed_dataset["train"]) * 0.5))), # 50% der Daten
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collate_function,
    compute_metrics=compute_metrics
)

trainer_esm_final.train()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


In [72]:

# Optimierte TrainingArguments gegen Overfitting
from transformers import TrainingArguments
import os
import datetime

BATCH_SIZE = 32
DATA_LOADER_NUM_WORKERS = 2
NUM_EPOCHS = 10

# Konfiguration für stabileres Fine-Tuning und bessere Generalisierung
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1

DATE = datetime.date.today()
OUTPUT_DIR = Path(demo_path/f"CAFA6_ESM_Aspects_Optimized")

training_args_optimized = TrainingArguments(
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    max_grad_norm = MAX_GRAD_NORM,
    warmup_ratio = WARMUP_RATIO,
    lr_scheduler_type = "cosine",
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs = NUM_EPOCHS,
    logging_strategy = "epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=DATA_LOADER_NUM_WORKERS,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    push_to_hub=False,
    remove_unused_columns=False
)

print("Optimierte TrainingArguments wurden definiert.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Optimierte TrainingArguments wurden definiert.


In [ ]:
def check_batch_integrity(dataset_dict):
    # Use .unique() to access column data without triggering the row-level transform
    train_prots = set(dataset_dict['train'].unique('protein'))
    val_prots = set(dataset_dict['validation'].unique('protein'))
    test_prots = set(dataset_dict['test'].unique('protein'))

    overlap_train_val = train_prots.intersection(val_prots)
    overlap_train_test = train_prots.intersection(test_prots)

    print(f"Anzahl Proteine - Train: {len(train_prots)}, Val: {len(val_prots)}, Test: {len(test_prots)}")
    print(f"Schnittmenge Train/Val: {len(overlap_train_val)}")
    print(f"Schnittmenge Train/Test: {len(overlap_train_test)}")

    if len(overlap_train_val) == 0:
        print("✅ Erfolg: Keine Überschneidungen zwischen Train und Validation.")
    else:
        print(f"❌ Fehler: Es gibt {len(overlap_train_val)} Überschneidungen!")

check_batch_integrity(processed_dataset)

Anzahl Proteine - Train: 41871, Val: 17930, Test: 14980
Schnittmenge Train/Val: 0
Schnittmenge Train/Test: 0
✅ Erfolg: Keine Überschneidungen zwischen Train und Validation.


#### 3.5.1 ESM based Sequence of Experts

The notion being, that when by still using the graph structure of the output data, but using mans partially fine-tuned Models, that will receive the previous classification results as an additional input to the classification layers, should have a better signal from the previous classifications, ideally leading to higher accuracys. While re-using most of the Pre-trained NN many times - technically running it once, and then feeding the data forward each time through the fine-tuned last layers and classifier heads.

Things to figure out:
- Does this work?
- How to inject the predictions
- Prepare the data with multiple layers of labels
- how to handle contradicting labels? one GO term might be at different deophs in the graph.
- re-assembling the data
- Is this actually bettern than just running the predicions individually and using statistics (I.e. if an animal was predicted cold-blodded, mammal, equine, horse - colg-blodded is likely wrong), which predicted nodes are likely false, etc.

##4. Model uploading...


In [ ]:
from google.colab import userdata
# Push our model to the Hugging Face Hub
model_on_hub_url = trainer.push_to_hub(commit_message="Uploaded trained facebook/esm2_t30_150M_UR50D on small set for [CFP] Prediction, it overfitted.",
                                                token=userdata.get('Huggingface')
                                                )
print(f"[INFO] Model uploaded to HuggingFace hub with {model_on_hub_url}")

AttributeError: 'ProtInteractionClassifierV2' object has no attribute 'config'